# 07c — Compression as Philosophy

## The One-Sentence Version
Intelligence was never about processing everything — it's about knowing what to cut.

## What You'll Build Intuition For
- Why compression is a fundamental principle, not just a technique
- How engineering, science, and thinking all reduce dimensions
- The connection between parsimony, Occam's razor, and PCA
- What this means for your own work

## Prerequisites
- The entire course — this notebook is the capstone

## The Story

Andrew Ng made an observation that stuck: the companies that win aren't the ones with the most data or the biggest compute budget. They're the ones that find the most truth in the least data. Brute force is for whoever has the deepest pockets. Compression — real understanding — is for everyone else.

This isn't just about machine learning. It's about how we think. Newton didn't catalogue every falling object — he compressed infinite observations into $F = ma$. The periodic table compressed infinite materials into ~100 elements. A good medical diagnosis compresses thousands of symptoms and test results into a single actionable conclusion.

Dimensionality reduction isn't a technique. It's a worldview.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from utils.plotting import apply_style, COLOURS

apply_style()

## The Ng Argument

The core insight:

> "The ability to process data is getting cheaper. The ability to know *which* data matters is getting more valuable."

- **Brute force** = throw compute at everything. Expensive, fragile, doesn't generalise.
- **Compression** = find the structure, discard the noise, work with what matters.

Every method we've learned is a form of this:
- PCA: find the directions that matter, ignore the rest
- Feature selection: find the variables that matter, ignore the rest
- Autoencoders: learn what to keep through a bottleneck
- Sensitivity analysis: find the parameters that matter, fix the rest

In [ ]:
# Visualise: compression in engineering
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Fourier: signal → frequencies
t = np.linspace(0, 1, 1000)
signal = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 12 * t)
signal_noisy = signal + np.random.default_rng(42).normal(0, 0.3, len(t))

freqs = np.fft.rfftfreq(len(t), d=t[1] - t[0])
spectrum = np.abs(np.fft.rfft(signal_noisy))

axes[0].plot(freqs[:60], spectrum[:60], color=COLOURS["signal"], linewidth=1.5)
axes[0].axvline(x=5, color=COLOURS["accent"], linestyle="--", alpha=0.7)
axes[0].axvline(x=12, color=COLOURS["accent"], linestyle="--", alpha=0.7)
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Fourier Transform\n1000 samples → 2 frequencies")

# 2. Image compression: explained variance
components = np.arange(1, 65)
# Simulate typical explained variance curve for an image
cumvar = 1 - np.exp(-components / 8)
axes[1].plot(components, cumvar, color=COLOURS["signal"], linewidth=2)
axes[1].axhline(y=0.95, color=COLOURS["accent"], linestyle="--", label="95%")
axes[1].axvline(x=15, color=COLOURS["noise"], linestyle=":", alpha=0.7)
axes[1].set_xlabel("Components kept")
axes[1].set_ylabel("Information retained")
axes[1].set_title("Image Compression\n64 pixels → 15 components")
axes[1].legend()

# 3. Science: laws compress observations
g = 9.81
t_fall = np.linspace(0, 3, 50)
h_data = 0.5 * g * t_fall**2 + np.random.default_rng(42).normal(0, 1, 50)
h_model = 0.5 * g * t_fall**2

axes[2].scatter(t_fall, h_data, s=15, alpha=0.5, color=COLOURS["noise"], label="Observations")
axes[2].plot(t_fall, h_model, color=COLOURS["signal"], linewidth=2, label="$h = \\frac{1}{2}gt^2$")
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Distance (m)")
axes[2].set_title("Newton's Law\n50 observations → 1 equation")
axes[2].legend()

fig.suptitle("Compression is everywhere: engineering, images, science", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/07c_compression_everywhere.png", dpi=150, bbox_inches="tight")
plt.show()

print("Fourier: any signal → a few dominant frequencies")
print("JPEG: millions of pixels → perceptually relevant information only")
print("Newton: infinite observations → F = ma")
print("\nAll of engineering and science is finding the right subspace.")

## Compression in Science

Science IS dimensionality reduction applied to reality.

| Domain | Raw Dimensions | Compressed Form | Reduction |
|--------|---------------|-----------------|----------|
| **Mechanics** | Infinite possible motions | $F = ma$ | ∞ → 1 equation |
| **Chemistry** | Infinite materials | ~100 elements | ∞ → 100 |
| **Genetics** | 3 billion base pairs | ~20,000 genes | 3B → 20K |
| **Thermodynamics** | 10²³ particle states | 3 variables (T, P, V) | 10²³ → 3 |
| **Epidemiology** | Millions of cases | $R_0$, $\gamma$, $\beta$ | 10⁶ → 3 |

Every scientific law is a compression: it captures the essential structure and discards the noise. The periodic table is PCA on chemistry. $F = ma$ is a rank-1 approximation of mechanical behaviour.

## Compression in Thinking

A good model isn't one that includes everything. It's one that includes the right things and nothing else.

This principle has many names:

- **Occam's Razor:** Among competing explanations, prefer the simplest
- **Parsimony:** Use the minimum number of assumptions needed
- **Minimum Description Length (MDL):** The best model minimises total description (model + data given model)
- **Feynman:** "What I cannot create, I do not understand"

Notice that Feynman's principle is about *compression*. To create something, you must understand its essential structure — the signal, not the noise. You must know what matters and what doesn't.

In [ ]:
# The parsimony curve: model complexity vs total description length
complexity = np.linspace(1, 50, 100)
model_cost = complexity * 0.5  # model description grows with complexity
data_cost = 50 * np.exp(-complexity / 8) + 2  # residual error shrinks but bottoms out
total = model_cost + data_cost

optimal = complexity[np.argmin(total)]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(complexity, model_cost, '--', color=COLOURS["accent"], linewidth=2,
        label="Model complexity")
ax.plot(complexity, data_cost, '--', color=COLOURS["noise"], linewidth=2,
        label="Residual error")
ax.plot(complexity, total, '-', color=COLOURS["signal"], linewidth=3,
        label="Total description length")
ax.axvline(x=optimal, color=COLOURS["success"], linestyle=':', linewidth=2,
           label=f"Optimal complexity ≈ {optimal:.0f}")
ax.set_xlabel("Model Complexity (number of parameters)")
ax.set_ylabel("Description Length")
ax.set_title("The Parsimony Principle: simplest model that captures the structure")
ax.legend()
plt.tight_layout()
plt.savefig("visuals/07c_parsimony.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Too simple (left): high residual error — model misses the structure.")
print(f"Too complex (right): model memorises noise — doesn't generalise.")
print(f"Just right (≈{optimal:.0f} parameters): captures the signal, discards the noise.")
print(f"\nThis IS the signal-not-noise principle, stated mathematically.")

## Bringing It Home

Every dataset, every simulation, every project has the same structure:

- **A high-dimensional space** of possible inputs, parameters, or features
- **A low-dimensional subspace** where the actual signal lives
- **Noise** everywhere else

Your job — whether you're a data scientist, an engineer, a clinician, or a researcher — is to find that subspace. To separate signal from noise. To compress.

The questions to always ask:

1. **What are the 3-5 dimensions that actually matter?**
2. **In your simulation: which parameters drive the outcome?**
3. **In your data: where does the signal live?**

The answer to these questions is worth more than any algorithm.

In [ ]:
# Final visual: the journey from Module 01 to Module 07
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)
ax.set_xticks([])
ax.set_yticks([])

modules = [
    ("01", "What is a\nDimension?", COLOURS["signal"]),
    ("02", "Redundancy &\nStructure", COLOURS["accent"]),
    ("03", "Linear\nMethods", COLOURS["success"]),
    ("04", "Nonlinear\nMethods", "#8B5CF6"),
    ("05", "Feature\nSelection", "#EC4899"),
    ("06", "Learned\nCompression", "#06B6D4"),
    ("07", "Applied\nThinking", COLOURS["noise"]),
]

for i, (num, label, colour) in enumerate(modules):
    x = 0.8 + i * 1.3
    rect = plt.Rectangle((x - 0.5, 1.2), 1.0, 1.6, facecolor=colour, alpha=0.7,
                          edgecolor="white", linewidth=2, zorder=2)
    ax.add_patch(rect)
    ax.text(x, 2.4, num, ha="center", va="center", fontsize=16, fontweight="bold",
            color="white", zorder=3)
    ax.text(x, 1.7, label, ha="center", va="center", fontsize=8,
            color="white", zorder=3)
    if i < len(modules) - 1:
        ax.annotate("", xy=(x + 0.6, 2.0), xytext=(x + 0.5, 2.0),
                    arrowprops=dict(arrowstyle="->", color=COLOURS["neutral"],
                                    lw=2), zorder=1)

# Bottom text
ax.text(5, 0.5, "Signal, not noise.", ha="center", fontsize=20,
        fontweight="bold", color=COLOURS["signal"], style="italic")

ax.set_title("The Complete Journey", fontsize=14)
plt.tight_layout()
plt.savefig("visuals/07c_journey.png", dpi=150, bbox_inches="tight")
plt.show()

<details>
<summary><b>The Maths Behind This</b></summary>

**Minimum Description Length (MDL):**

$$\text{MDL}(M) = \underbrace{L(M)}_{\text{model complexity}} + \underbrace{L(D | M)}_{\text{data given model}}$$

This formalises Occam's Razor: the best explanation minimises total description length. A model that's too simple has high $L(D|M)$ (can't explain the data). A model that's too complex has high $L(M)$ (the explanation is longer than the data). The optimum is the compression sweet spot.

**Kolmogorov complexity:** The length of the shortest program that produces a given output. Incomputable in general, but MDL is its practical approximation. PCA, Lasso, and autoencoders all approximate Kolmogorov compression of data — finding shorter descriptions that preserve the essential structure.

**Connection to PCA:** PCA finds the rank-$k$ matrix $X_k$ that minimises $\|X - X_k\|_F^2$. The rank $k$ is the model complexity. The residual $\|X - X_k\|_F^2$ is the data unexplained by the model. Choosing $k$ via explained variance threshold is an informal MDL criterion.

</details>

## Where This Matters

**Everywhere.** This isn't a technique notebook — it's a thinking framework.

- A doctor diagnosing a patient is compressing thousands of data points into one actionable conclusion
- A commander planning an operation is reducing infinite possibilities to the 3-5 courses of action that matter
- A scientist writing a paper is compressing months of work into the essential findings
- An engineer designing a system is finding the low-dimensional subspace of designs that actually work

The tools change. The principle doesn't: **find the signal, discard the noise.**

## Feynman Check

1. **Give an example of dimensionality reduction from outside computer science.**

2. **Why might a simpler model outperform a more complex one?**

3. **What's the connection between Occam's razor and PCA?**

---

**This concludes signal-not-noise.**

You started with a single patient and a single measurement. You ended with the philosophical foundation of compression itself. Along the way, you built intuition for what dimensions are, why they cause problems, and how to tame them — with linear methods, nonlinear methods, feature selection, and neural networks.

The next time you face a high-dimensional problem, you'll know what to do: find the signal. Discard the noise. Start simple. Escalate only if needed.

That's the whole game.